# Imports

In [98]:
import boto3
import os
import sagemaker
from sagemaker.core import image_uris
from sagemaker.core.processing import ScriptProcessor
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.mlops.workflow.steps import ProcessingStep

# Initialization

In [99]:
def list_all_sagemaker_domains(region_name='eu-west-1'):
    """
    List all SageMaker domains in the current AWS account and region.

    Returns:
        list: A list of dictionaries containing domain details.
    """
    client = boto3.client('sagemaker',region_name=region_name)
    response = client.list_domains()
    return response.get('Domains', [])

def get_sagemaker_domain_arn(domain_name, region_name='eu-west-1'):
    """
    Get the ARN of a specific SageMaker domain by its name.

    Args:
        domain_name (str): The name of the SageMaker domain.
        region_name (str): The AWS region where the domain is located.

    Returns:
        str: The ARN of the specified SageMaker domain, or None if not found.
    """
    domains = list_all_sagemaker_domains(region_name)
    for domain in domains:
        if domain['DomainName'] == domain_name:
            return domain['DomainArn']
    return None

In [100]:
list_of_domains = list_all_sagemaker_domains()
print("List of SageMaker Domains:", list_of_domains)
print("ARN of the first domain:", get_sagemaker_domain_arn(list_of_domains[0]['DomainName']) if list_of_domains else "No domains found")

List of SageMaker Domains: [{'DomainArn': 'arn:aws:sagemaker:eu-west-1:746760141372:domain/d-mvykom11e1ij', 'DomainId': 'd-mvykom11e1ij', 'DomainName': 'QuickSetupDomain-20260924T193773', 'Status': 'InService', 'CreationTime': datetime.datetime(2026, 9, 24, 16, 37, 48, 627000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2026, 9, 24, 16, 38, 5, 153000, tzinfo=tzlocal()), 'Url': 'https://d-mvykom11e1ij.studio.eu-west-1.sagemaker.aws'}]
ARN of the first domain: arn:aws:sagemaker:eu-west-1:746760141372:domain/d-mvykom11e1ij


In [101]:
pipeline_session = PipelineSession()
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
account_id = boto3.client('sts').get_caller_identity().get('Account')
print(f"Region: {region}, Account ID: {account_id}")
role = f"arn:aws:iam::746760141372:role/service-role/AmazonSageMaker-ExecutionRole-20260924T193773"
domain_name = os.getenv('DOMAIN_NAME') 
print(f"Domain Name from environment variable: {domain_name}")
domain_arn = get_sagemaker_domain_arn(domain_name) if domain_name else None
print(f"Domain ARN: {domain_arn if domain_arn else 'Domain not found'}")
print(f"role: {role}")

Region: eu-west-1, Account ID: 746760141372
Domain Name from environment variable: QuickSetupDomain-20260924T193773


Domain ARN: arn:aws:sagemaker:eu-west-1:746760141372:domain/d-mvykom11e1ij
role: arn:aws:iam::746760141372:role/service-role/AmazonSageMaker-ExecutionRole-20260924T193773


# Pipeline Parameters

In [102]:
#S3
s3_bucket = ParameterString(name="S3Bucket", default_value=f"telco-churn-746760141372-eu-west-1-an")
s3_prefix = ParameterString(name="S3Prefix", default_value=f"input")
s3_key = ParameterString(name="S3Key", default_value=f"telco_churn.csv")


# Processor

In [103]:
script_processor = ScriptProcessor(
    command=["python3"],
    role=role,
    image_uri=image_uris.retrieve(
        framework="sklearn",
        region=region,
        version="1.2-1",
        instance_type="ml.t3.medium",
    ),
    instance_count=1,
    instance_type="ml.t3.medium",
    base_job_name="driftwatch-processing-job",
    sagemaker_session=pipeline_session,
)

[09/24/26 23:05:41] INFO     Defaulting to only available      ]8;id=15155738;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=15155739;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\
                             Python version: py3                                


# Step 1 (Preprocessing)

In [104]:
step_preprocessing = ProcessingStep(
    name="PreprocessingStep",
    step_args=script_processor.run(
        code="steps/preprocess/preprocess.py",
        arguments=[
            "--s3_bucket", s3_bucket,
            "--s3_prefix", s3_prefix,
            "--s3_key", s3_key,
            "--region", region,
        ],
    ),
)

# Pipeline Creation

In [105]:
pipeline_name = "DriftWatchPipeline"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[s3_bucket, s3_prefix, s3_key],
    steps=[step_preprocessing],
    sagemaker_session=pipeline_session,
)

In [106]:
pipeline.upsert(
    role_arn=role,
    tags = [
        {"Key": "sagemaker:domain-arn", "Value": domain_arn},
    ]
)

[09/24/26 23:05:42] WARNING  Popping out 'ProcessingJobName'    ]8;id=15155744;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=15155745;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             from the pipeline definition by                    
                             default since it will be                           
                             overridden at pipeline execution                   
                             time. Please utilize the                           
                             PipelineDefinitionConfig to                        
                             persist this field in the pipeline                 
                             definition if desired.                             
                    WARNING  Popping out 'ProcessingJobName'    ]8;id=15155750;file:///opt/conda/lib/python3.12/site-packages/sagem

{'PipelineArn': 'arn:aws:sagemaker:eu-west-1:746760141372:pipeline/DriftWatchPipeline',
 'PipelineVersionId': 1,
 'ResponseMetadata': {'RequestId': 'aeba7d38-a1cf-46bb-a5bd-44c27a55fb02',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'aeba7d38-a1cf-46bb-a5bd-44c27a55fb02',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '108',
   'date': 'Thu, 24 Sep 2026 23:05:42 GMT'},
  'RetryAttempts': 0}}

In [107]:
pipeline.start()

PipelineExecution(arn='arn:aws:sagemaker:eu-west-1:746760141372:pipeline/DriftWatchPipeline/execution/uy9n52gumede', sagemaker_session=<sagemaker.core.workflow.pipeline_context.PipelineSession object at 0x7f9a453dac00>)